# Load model

In [78]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from itertools import islice

In [79]:
import sys
sys.path.append('/ceph/submit/data/user/k/kyoon/KYoonStudy/ssm_regression/modules')
sys.path.append('/ceph/submit/data/user/k/kyoon/KYoonStudy/ssm_regression/ligo')
from models import S4Model
from data_bns import get_dataloaders

In [80]:
ssm_model_path = '/ceph/submit/data/user/k/kyoon/KYoonStudy/models/BNS/output/model.SSM.BNS.NLLGaussian.d16.n10.250714214116.path'
hdf5_path = '/ceph/submit/data/user/k/kyoon/KYoonStudy/models/BNS/bns_waveforms.hdf5'
split_indices_file = '/ceph/submit/data/user/k/kyoon/KYoonStudy/models/BNS/bns_data_indices.npz'

In [81]:
ssm_model = S4Model(d_input=2, d_output=2, d_model=16, n_layers=10, loss='NLLGaussian', dropout=0.0, prenorm=False)
ssm_model = ssm_model.to(device)
ssm_model.load_state_dict(torch.load(ssm_model_path, map_location=device))
ssm_model.eval()

/tmp/ipykernel_1673827/4290684304.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ssm_model.load_state_dict(torch.load(ssm_model_path, map_location=device))


S4Model(
  (encoder): Linear(in_features=2, out_features=16, bias=True)
  (s4_layers): ModuleList(
    (0-9): 10 x S4D(
      (kernel): S4DKernel()
      (activation): GELU(approximate='none')
      (dropout): Identity()
      (output_linear): Sequential(
        (0): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
        (1): GLU(dim=-2)
      )
    )
  )
  (norms): ModuleList(
    (0-9): 10 x LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  )
  (dropouts): ModuleList(
    (0-9): 10 x Dropout1d(p=0.0, inplace=False)
  )
  (decoder): Linear(in_features=16, out_features=2, bias=True)
)

In [82]:
_, _, test_dataloader = get_dataloaders(
    hdf5_path, downsample_factor=2, duration=4, scale_factor=1.,
    train_batch_size=1, val_batch_size=1, test_batch_size=1,
    train_split=0.8, test_split=0.1,
    split_indices_file='', random_seed=42)

In [83]:
next(iter(test_dataloader)) # inspect

[tensor([[3.6575e-24, 8.2357e-24, 1.1488e-23,  ..., 2.3754e-26, 2.8743e-26,
          3.1256e-26]]),
 tensor([[ 1.6008e-23,  1.7343e-23,  1.5883e-23,  ..., -2.8813e-26,
          -1.8373e-26, -6.3817e-27]]),
 {'a_1': tensor([0.]),
  'a_2': tensor([0.]),
  'dec': tensor([-0.3703]),
  'ifo_snrs': tensor([[ 6.2114, 10.1019]]),
  'mass_1': tensor([2.3532]),
  'mass_2': tensor([2.1517]),
  'phase': tensor([1.9939]),
  'phi_12': tensor([0.]),
  'phi_jl': tensor([0.]),
  'psi': tensor([0.]),
  'ra': tensor([1.8939]),
  'redshift': tensor([0.0388]),
  'snr': tensor([11.8588]),
  'theta_jn': tensor([1.1863]),
  'tilt_1': tensor([0.]),
  'tilt_2': tensor([0.])},
 tensor([9])]

In [94]:
# Create iterator
it = iter(test_dataloader)

# Skip first 12 batches and get the next one
h1, l1, d, idx = next(islice(it, 12, None))
inputs = torch.stack([h1.to(device), l1.to(device)], dim=2)
inputs

tensor([[[ 8.0257e-24,  4.8425e-24],
         [ 7.2330e-24,  1.5218e-24],
         [ 5.2896e-24, -2.0415e-24],
         ...,
         [ 4.5820e-27, -1.9676e-26],
         [-2.5480e-27, -1.4822e-26],
         [-9.3231e-27, -8.0445e-27]]])

In [95]:
ssm_model(inputs*1e22)

tensor([[1.9840e+00, 1.0101e-05]], grad_fn=<CatBackward0>)

In [96]:
# Skip first 6 batches and get the next one
h1, l1, d, idx = next(islice(it, 6, None))
inputs = torch.stack([h1.to(device), l1.to(device)], dim=2)
inputs

tensor([[[ 1.0133e-23,  1.2263e-23],
         [ 6.7008e-24,  1.4017e-23],
         [ 1.6983e-24,  1.2486e-23],
         ...,
         [-3.9081e-24,  6.5553e-24],
         [-4.4046e-24,  6.5091e-24],
         [-4.7939e-24,  6.3045e-24]]])

In [97]:
ssm_model(inputs*1e22)

tensor([[1.4540e+00, 1.1062e-05]], grad_fn=<CatBackward0>)

In [88]:
print(inputs.abs().min())
print(inputs.abs().max())

tensor(3.0229e-27)
tensor(7.1739e-23)


In [89]:
mean = inputs.mean()
std = inputs.std()
inputs_rescaled = (inputs - mean) / (std)

In [90]:
inputs_rescaled

tensor([[[ 0.9350,  1.1321],
         [ 0.6173,  1.2945],
         [ 0.1543,  1.1527],
         ...,
         [-0.3646,  0.6038],
         [-0.4106,  0.5996],
         [-0.4466,  0.5806]]])

In [91]:
print('Mean:', inputs_rescaled.mean().item())
print('Std:', inputs_rescaled.std().item())

Mean: -1.3969838619232178e-09
Std: 1.0


In [92]:
print(inputs_rescaled.abs().min())
print(inputs_rescaled.abs().max())

tensor(9.7781e-05)
tensor(6.6369)
